# Day 36: CSV and JSON

**Today's question:** how do we save and load *structured* data (tables, records, nested lists) so that another
program, a spreadsheet, or our own program tomorrow can read it back correctly?

Before running a cell marked 🤔, predict what it will show.

In [ ]:
from pathlib import Path

out_dir = Path("output")        # everything this notebook writes goes into output/
out_dir.mkdir(exist_ok=True)
purchases_path = Path("data/purchases.csv")

## 1. Why not just split each line at the commas?

A **CSV** file (comma-separated values) is a table stored as plain text: one line per row, commas between columns.
Let's look at the raw text of a small one.

In [ ]:
print(purchases_path.read_text(encoding="utf-8"))

On Day 23 we read files line by line and used `.split(",")`.
🤔 **Predict:** how many pieces will each line split into? (The header has 3 columns.)

In [ ]:
with open(purchases_path, encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(",")
        print(len(parts), parts)

The comma inside `"notebooks, pens and folders"` is **part of the data**, not a separator. CSV protects it with
quotes, but `split(",")` knows nothing about quotes. So "the amount is the third piece" is wrong for some lines:

In [ ]:
with open(purchases_path, encoding="utf-8") as f:
    lines = f.read().strip().split("\n")

total = 0
for line in lines[1:]:          # skip the header line
    parts = line.split(",")
    total += float(parts[2])    # the third piece is NOT always the amount
print(total)

Python's built-in **`csv` module** knows all the CSV rules: quotes, commas inside quotes, even line breaks inside a
field. We never split CSV lines by hand.

## 2. `csv.reader`: each row becomes a list

In [ ]:
import csv

with open(purchases_path, newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)

- Every row is a **list of strings**. The quotes are gone and the comma stays inside the description.
- The header is just the first row.
- `newline=""` lets the `csv` module handle line endings itself. **Always** use it when you open a file for
  `csv` (reading or writing). Without it, files written on Windows get an empty line between rows.

🤔 **Predict:** what does this print?

In [ ]:
print(row[2], type(row[2]))

CSV files only contain **text**. Converting `"18.30"` to a number is our job.

In [ ]:
with open(purchases_path, newline="", encoding="utf-8") as f:
    rows = list(csv.reader(f))   # read all rows while the file is open

header = rows[0]
total = 0
for row in rows[1:]:
    total += float(row[2])
print(header)
print(f"Total: {total:.2f}")

## 3. `csv.DictReader`: each row becomes a dict

`row[2]` is hard to read: which column is number 2? And if someone adds a column, every index shifts.
`DictReader` uses the **header line as keys**, so we write `row["amount"]` instead.

🤔 **Predict:** the file has 6 lines. How many rows will be printed?

In [ ]:
with open(purchases_path, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row)

Five: the header line becomes the keys. The usual pattern is to load the whole file into a **list of dicts**
(Day 08), then work with the list after the file is closed.

In [ ]:
with open(purchases_path, newline="", encoding="utf-8") as f:
    purchases = list(csv.DictReader(f))

print(len(purchases), "purchases")
for purchase in purchases:
    amount = float(purchase["amount"])
    print(f"{purchase['date']}  {purchase['description']:<28} {amount:>6.2f}")

## 4. Writing CSV files

`csv.writer` writes lists. `writerow` writes one row; `writerows` writes a list of rows.

In [ ]:
attendance = [
    ["name", "present", "late"],
    ["Asha", 18, 1],
    ["Chen, Mei", 20, 0],
    ["Bilal", 15, 4],
]
attendance_path = out_dir / "attendance_summary.csv"
with open(attendance_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(attendance)

🤔 **Predict:** what will the `"Chen, Mei"` line look like inside the file?

In [ ]:
print(attendance_path.read_text(encoding="utf-8"))

The writer added the quotes for us. When our data is already a **list of dicts**, `csv.DictWriter` is easier:
tell it the column order with `fieldnames`, then call `writeheader()` once.

In [ ]:
results = [
    {"name": "Sara", "score": 88, "passed": True},
    {"name": "Omar", "score": 47, "passed": False},
    {"name": "Zara", "score": 73, "passed": True},
]
results_path = out_dir / "results.csv"
with open(results_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "score", "passed"])
    writer.writeheader()          # first line: name,score,passed
    writer.writerows(results)

print(results_path.read_text(encoding="utf-8"))

🤔 **Predict:** we read Omar's row back. What type is `score`, and is `passed == False`?

In [ ]:
with open(results_path, newline="", encoding="utf-8") as f:
    loaded = list(csv.DictReader(f))

print(loaded[1])
print(type(loaded[1]["score"]), loaded[1]["passed"] == False)

`"False"` (a string) is not `False` (a bool). **CSV forgets types**: everything comes back as text.

## 5. JSON: Python data as text

**JSON** is a text format that looks almost exactly like Python dicts and lists. It keeps numbers as numbers,
`true`/`false`, `null` (Python's `None`) and **nesting** (lists inside dicts inside lists). Most web services,
including the AI models in Week 11, send their answers as JSON.

🤔 **Predict:** how will `True` and `None` be written in JSON?

In [ ]:
import json

profile = {
    "name": "Fatima",
    "age": 21,
    "courses": ["Python", "Statistics"],
    "has_laptop": True,
    "mentor": None,
}
text = json.dumps(profile, indent=2)
print(text)
print(type(text))

`json.dumps` turned the dict into a **string**. `indent=2` makes it readable for humans.
To work with **files**, use `json.dump` (save) and `json.load` (read back):

In [ ]:
profile_path = out_dir / "profile.json"
with open(profile_path, "w", encoding="utf-8") as f:
    json.dump(profile, f, indent=2)       # data first, then the file

with open(profile_path, encoding="utf-8") as f:
    loaded_profile = json.load(f)

print(loaded_profile == profile)
print(type(loaded_profile["age"]), type(loaded_profile["has_laptop"]))

The types survived the round trip. How to remember the four functions: **the `s` means string**.

| | with a file | with a string |
|-|-------------|---------------|
| Python → JSON (save) | `json.dump(data, f)` | `json.dumps(data)` |
| JSON → Python (load) | `json.load(f)` | `json.loads(text)` |

In [ ]:
reply = '{"item": "pen", "quantity": 3, "in_stock": true}'
order = json.loads(reply)
print(order["quantity"] * 2, order["in_stock"])

🤔 **Predict:** JSON is *almost* Python. What comes back after this round trip?

In [ ]:
scores = {1: (90, 85), 2: (70, 75)}
round_trip = json.loads(json.dumps(scores))
print(round_trip)

JSON keys are **always strings**, and JSON has no tuples (they become lists). Use string keys and lists in data you
plan to save, and nothing will surprise you.

## 6. CSV or JSON?

| Choose **CSV** when… | Choose **JSON** when… |
|----------------------|-----------------------|
| the data is one table: every row has the same columns | the data is nested (a recipe with a list of ingredients) |
| people will open it in a spreadsheet | you want numbers, booleans and `None` back with their types |
| it is a log or export that keeps growing row by row | it is settings, a saved app state, or an API reply |

## 7. Common mistakes (these cells fail **on purpose**)

In [ ]:
# Mistake 1: forgetting that CSV values are text
total = 0
for purchase in purchases:
    total += purchase["amount"]

In [ ]:
# Mistake 2: the key must match the header exactly (case matters)
print(purchases[0]["Amount"])

In [ ]:
# Mistake 3: arguments in the wrong order. It is json.dump(data, file)
with open(out_dir / "settings.json", "w", encoding="utf-8") as f:
    json.dump(f, profile)

In [ ]:
# Mistake 4: sets cannot be saved as JSON (use a list instead)
json.dumps({"name": "Ken", "skills": {"python", "sql"}})

In [ ]:
# Mistake 5: JSON text needs double quotes, not single quotes
json.loads("{'name': 'Ken'}")

A missing or damaged file should not crash a program. `json.JSONDecodeError` is the error for broken JSON
(it is a kind of `ValueError`), so we catch it like any other exception from Day 25:

In [ ]:
def load_json(path, default):
    """Return the data saved in a JSON file, or default if the file is missing or broken."""
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return default
    except json.JSONDecodeError:
        print(f"Warning: {path} is not valid JSON, using the default.")
        return default


print(load_json(out_dir / "no_such_file.json", {}))
print(load_json(profile_path, {})["name"])

## 8. Real-world example: expenses CSV → JSON category summary

The course dataset `expenses.csv` holds three months of spending. Goal: one JSON file with the total, count and
average per category. We split the job into small functions (Day 19), which is tomorrow's topic.

In [ ]:
expenses_path = Path("../../../datasets/expenses.csv")


def load_expenses(path):
    """Read the expenses CSV and return a list of dicts with amount as a float."""
    with open(path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    for row in rows:
        row["amount"] = float(row["amount"])
    return rows


expenses = load_expenses(expenses_path)
print(len(expenses), "expenses")
print(expenses[0])

In [ ]:
def summarise_by_category(expenses):
    """Return {category: {"total": ..., "count": ..., "average": ...}}."""
    summary = {}
    for expense in expenses:
        category = expense["category"]
        if category not in summary:
            summary[category] = {"total": 0.0, "count": 0}
        summary[category]["total"] += expense["amount"]
        summary[category]["count"] += 1
    for info in summary.values():
        info["average"] = round(info["total"] / info["count"], 2)
        info["total"] = round(info["total"], 2)
    return summary

🤔 **Predict:** which category has the highest total?

In [ ]:
summary = summarise_by_category(expenses)
for category, info in summary.items():
    print(f"{category:<14} {info['count']:>3} items   total {info['total']:>8.2f}   average {info['average']:>7.2f}")

In [ ]:
summary_path = out_dir / "expenses_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(summary_path.read_text(encoding="utf-8"))

Any program can now read the summary back, with numbers still numbers:

In [ ]:
saved = load_json(summary_path, {})
print(saved["rent"]["total"], saved["food"]["count"])

## Recap

1. Never `split(",")` a CSV line. The `csv` module handles quotes. Open files with `newline=""` and `encoding="utf-8"`.
2. `csv.DictReader` gives a list of dicts keyed by the header. **Every value is a string**: convert it.
3. `csv.DictWriter(f, fieldnames=[...])` + `writeheader()` + `writerows(list_of_dicts)` writes a table.
4. `json.dump(data, f, indent=2)` / `json.load(f)` save and load Python data with its types. `dumps`/`loads` work
   with strings. JSON keys are always strings.
5. CSV for flat tables and spreadsheets; JSON for nested data, settings and app state.

**Next time:** we turn today's script into a proper **pipeline**: load → clean → analyse → report, one small
function per step, each one tested on its own.